# AI Drug Discovery with BioNeMo

This notebook is the code reference for the agent workflow. During the workshop, run the GPU resource checks first, then follow [BioNeMo_with_Agent.md](BioNeMo_with_Agent.md). **Run All** provides an optional complete reference run; let it finish before making model calls through the agent. See [README.md](README.md) for the course sequence.


## Before Starting: Check GPU Resources (Workshop VM Only)

**For this workshop VM only:** the courses share one GPU. Run one GPU course at a time. The rest of the lesson can also be used in your own environment.

Save the previous notebook, then select **Kernel → Shut Down Kernel**. Closing the tab does not stop the kernel. Run this cell to check GPU use:


In [ ]:
import shutil, subprocess
if shutil.which("bootcamp-resources"):
    subprocess.run(["bootcamp-resources", "status"], check=True)
else:
    print("Self-study: check nvidia-smi and the NIM readiness URLs in README.md.")


Before starting, confirm that no other course kernel is running. If BioNeMo is stopped, `gpu_processes` must be empty. If the GPU is busy, shut down the previous course kernel and check again. Ask the instructor if other processes remain.

**Start BioNeMo** (skip this cell if all three models are already healthy):


In [ ]:
import shutil, subprocess
if shutil.which("bootcamp-resources"):
    subprocess.run(["bootcamp-resources", "start"], check=True)
else:
    print("Self-study: start your own NIMs following README.md before continuing.")


Check status again. Wait for GenMol, DiffDock, and Boltz-2 to show **healthy** before continuing. Rerun the status cell while the models start.


In [ ]:
import shutil, subprocess
if shutil.which("bootcamp-resources"):
    subprocess.run(["bootcamp-resources", "status"], check=True)
else:
    print("Self-study: check nvidia-smi and the NIM readiness URLs in README.md.")


Before switching courses, use **Release GPU Resources** at the end of this notebook. Details: `00_START_HERE.txt` in the workshop's main directory.


## AI for Virtual Screening and Lead Optimization

Drug discovery seeks molecules that act on a biological target and have suitable properties for development. **Virtual screening** ranks molecules computationally before experimental testing. **Lead optimization** modifies an active molecule to improve potency, selectivity, and other properties.

Testing every possible molecule is impractical. AI models learn from chemical and structural data to generate molecules, predict binding poses, and estimate affinity. Together, these capabilities help prioritize molecules for synthesis and testing and guide the next design cycle.

This workflow demonstrates one design-and-screening cycle with three BioNeMo NIM models:

| Model | Task | Result to inspect |
|---|---|---|
| GenMol | Generate analogues around a retained core | Molecular structures and properties |
| DiffDock | Dock selected molecules to the target protein | Binding poses and pose confidence |
| Boltz-2 | Evaluate one selected analogue | Predicted complex and affinity |


## Example: EGFR and Gefitinib

**EGFR** (epidermal growth factor receptor) is a receptor tyrosine kinase that controls cell-growth signaling. Activating mutations in EGFR can drive non-small cell lung cancer. **Gefitinib** (Iressa) inhibits EGFR kinase activity and is used in lung cancers with sensitizing EGFR mutations. [Background: NCI](https://www.cancer.gov/about-cancer/treatment/drugs/gefitinib)

The experimental structure [PDB 2ITY](https://www.rcsb.org/structure/2ITY) shows gefitinib bound to the **wild-type EGFR kinase domain**. It provides both a protein structure for docking and a known ligand pose for comparison.

Starting from this pair, explore related molecules through three tasks:

| Step | Biological question | Model output |
|---|---|---|
| GenMol | What related molecules can we generate? | Molecular structures and properties |
| DiffDock | How might selected molecules fit into EGFR? | Ranked binding poses |
| Boltz-2 | What complex and affinity does the model predict for one candidate? | Predicted complex and affinity scores |

This notebook runs the workflow directly in Python. Generate 20 molecules, dock up to three eligible analogues alongside gefitinib, and evaluate one analogue with Boltz-2.

## 1. Retrieve the Protein and Small Molecules from Database

### Load the Libraries and Output Locations

RDKit handles molecular structures, pandas displays result tables, and requests communicates with the NIM APIs. The shared utilities save requests and results and provide the predefined molecular viewer.

Keep `USE_PREPARED = False` for live inference. Set it to `True` to use the saved example pathway. Inputs go to `data/`, run results to `results/manual/`, and visualizations to `figures/`.

In [ ]:
from pathlib import Path
import json, os
import requests
import pandas as pd
from IPython.display import display
from rdkit import Chem
from rdkit.Chem import AllChem, Draw
from Bio.SeqUtils import seq1
from support import workshop as w

ROOT = w.ROOT
DATA, RESULTS, FIGURES, PREPARED = [ROOT / p for p in
    ('data', 'results/manual', 'figures', 'prepared')]
for folder in (DATA, RESULTS, FIGURES, PREPARED):
    folder.mkdir(parents=True, exist_ok=True)
USE_PREPARED = False  # Your choice: use the supplied matching example pathway.
pd.Series(w.ENDPOINTS, name='Local NIM endpoints').to_frame()


### Retrieve EGFR and Gefitinib

Download the **2ITY** protein–ligand complex and the chemical definition of **gefitinib (ligand code IRE)** from RCSB PDB.

The code separates EGFR chain A from its bound ligand. It keeps gefitinib's experimental coordinates and uses its chemical definition to restore bond orders. The PDB `SEQRES` records provide the complete 327-residue construct for Boltz-2; the `ATOM` records provide the experimentally resolved protein coordinates for docking.

In [ ]:
sources = {
    '2ITY.pdb': 'https://files.rcsb.org/download/2ITY.pdb',
    'IRE_ideal.sdf': 'https://files.rcsb.org/ligands/download/IRE_ideal.sdf',
}
for name, url in sources.items():
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    (DATA / name).write_bytes(response.content)

lines = (DATA / '2ITY.pdb').read_text().splitlines()
protein_lines = [line for line in lines if line.startswith('ATOM  ')
                 and line[21] == 'A' and line[16] in ' A']
receptor_pdb = '\n'.join(protein_lines) + '\nEND\n'
(DATA / 'egfr_receptor.pdb').write_text(receptor_pdb)
sequence = ''.join(seq1(residue) for line in lines
    if line.startswith('SEQRES') and line[11] == 'A'
    for residue in line[19:70].split())
assert len(sequence) == 327
(DATA / 'egfr_sequence.txt').write_text(sequence + '\n')
(DATA / 'egfr_sequence.fasta').write_text('>2ITY_A EGFR kinase domain\n' + sequence + '\n')
ligand_lines = [line for line in lines if line.startswith('HETATM')
    and line[17:20] == 'IRE' and line[21] == 'A' and line[16] in ' A']
ligand = Chem.MolFromPDBBlock('\n'.join(ligand_lines) + '\nEND\n',
                            sanitize=False, removeHs=True)
template = Chem.MolFromMolFile(str(DATA / 'IRE_ideal.sdf'), removeHs=True)
ligand = AllChem.AssignBondOrdersFromTemplate(template, ligand)
reference = w.molecule(Chem.MolToSmiles(ligand))
reference_sdf = Chem.MolToMolBlock(ligand)
(DATA / 'gefitinib_reference.sdf').write_text(reference_sdf)
w.save_json(DATA / 'reference_molecule.json', reference)
w.save_json(DATA / 'sources.json', sources)
assert ligand.GetNumHeavyAtoms() == 31


### Visualize Gefitinib and Inspect Its Properties

Use **pandas** for the property table, **IPython** to display it in the notebook, and **RDKit** to draw gefitinib's 2D chemical structure.

| Command | What it does |
|---|---|
| `pd.DataFrame([dict(reference, name='Gefitinib')])` | Builds a one-row table from the saved molecular properties. |
| `display(...)` | Renders the table in the notebook. |
| `Draw.MolsToGridImage([ligand], legends=['Gefitinib'], subImgSize=(400, 260))` | Draws the molecule with a label in a 400 × 260 pixel panel. |

`reference` contains the molecular properties; `ligand` is the RDKit molecule prepared above. The drawing shows atom connectivity and bond types. The final expression in the cell displays the image automatically.

In [ ]:
display(pd.DataFrame([dict(reference, name='Gefitinib')]))
Draw.MolsToGridImage([ligand], legends=['Gefitinib'], subImgSize=(400, 260))


### Prepare the EGFR Alignment Once

A multiple sequence alignment (**MSA**) places related protein sequences together and supplies evolutionary information to Boltz-2. The alignment's first sequence must match the EGFR construct used here.

Reuse `data/egfr.a3m` if present. Otherwise, retrieve it with NVIDIA's hosted MSA-Search API using `NGC_API_KEY` from `.env`. The final line reports the protein length and number of alignment records.

In [ ]:
msa_file = DATA / 'egfr.a3m'
if not msa_file.exists():
    env_file = ROOT / '.env'
    if not env_file.is_file():
        raise RuntimeError('Missing course .env file with NGC_API_KEY.')
    key = ''
    for line in env_file.read_text().splitlines():
        name, separator, value = line.partition('=')
        if separator and name.strip() == 'NGC_API_KEY':
            key = value.strip().strip('"').strip("'")
    if not key:
        raise RuntimeError('Set NGC_API_KEY in the course .env file.')
    try:
        response = requests.post(
            'https://health.api.nvidia.com/v1/biology/colabfold/msa-search/predict',
            headers={'Authorization': 'Bearer ' + key},
            json={'sequence': sequence, 'databases': ['Uniref30_2302'],
                  'output_alignment_formats': ['a3m']}, timeout=900)
    finally:
        del key
    response.raise_for_status()
    msa_result = response.json()
    msa_file.write_text(next(iter(msa_result['alignments'].values()))['a3m']['alignment'])
    w.save_json(DATA / 'egfr_msa_search.json', msa_result)
msa = msa_file.read_text()
reference_payload = w.boltz_payload(reference['smiles'], sequence, msa)
print('EGFR:', len(sequence), 'residues; alignment records:', msa.count('>'))


## 2. Generate Small Molecule Analogues with GenMol

[NVIDIA Build](https://build.nvidia.com/nvidia/genmol-generate/deploy) · [GenMol 2.0 API](https://docs.nvidia.com/nim/bionemo/genmol/2.0.0/endpoints.html)

Give GenMol a gefitinib-derived core as a complete **SMILES** string, a text representation of molecular connectivity. The request asks for 20 molecules; `temperature` and `noise` control sampling, and `scoring='QED'` ranks drug-likeness.

The code checks the returned molecules, excludes invalid or duplicate structures, and retains molecules that preserve the core and have molecular weights of 250–550 g/mol. It selects up to three by QED and continues with fewer if needed.

Inspect the selection report and structure grid. **MW** is molecular weight, **cLogP** estimates lipophilicity, and **QED** summarizes drug-like properties on a 0–1 scale. The full report, including exclusion reasons, is saved as JSON and CSV.

In [ ]:
generation_payload = {
    'smiles': w.GENMOL_TEMPLATE,
    'num_molecules': 20,
    'temperature': 1.0,
    'noise': 1.0,
    'min_add_len': 24,
    'scoring': 'QED',
    'unique': True,
    'filter': False,
}
generation = (w.read_json(PREPARED / 'genmol.json') if USE_PREPARED else
              w.call_nim('genmol', generation_payload, RESULTS))

generated_count = len(generation['response'].get('molecules', []))
print('Requested:', generation_payload['num_molecules'], 'Returned:', generated_count)
report_path = RESULTS / 'generation_report.json'
selected = w.curate(generation['response'], reference['smiles'], limit=3,
                    report_path=report_path)
w.save_json(RESULTS / 'selected_candidates.json', selected)
report = pd.DataFrame(w.read_json(report_path))
report.to_csv(RESULTS / 'generation_report.csv', index=False)
print('Selected for docking:', len(selected), '(up to 3)')
display(report.drop(columns=['smiles'], errors='ignore').round(3))

if selected:
    grid_records = [dict(reference, candidate_id='Gefitinib')] + selected
    grid = Draw.MolsToGridImage(
        [Chem.MolFromSmiles(row['smiles']) for row in grid_records], molsPerRow=4,
        subImgSize=(300, 260), useSVG=False, returnPNG=False, legends=[
            f"{row['candidate_id']} | MW {row['MW']:.1f} | QED {row['QED']:.2f}"
            for row in grid_records])
    grid.save(str(FIGURES / 'genmol_molecules.png'))
    display(grid)
else:
    print('No eligible analogues remain. Review the report, generate a new batch, '
          'or choose prepared results. Downstream model calls will be skipped.')


## 3. Inspect Binding Poses with DiffDock

[NVIDIA Build](https://build.nvidia.com/mit/diffdock/deploy) · [API reference](https://docs.nvidia.com/nim/bionemo/diffdock/latest/api-reference.html)

Dock gefitinib and each selected analogue against the same experimental EGFR structure. The request supplies protein coordinates in PDB format and ligand SMILES as text (`ligand_file_type='txt'`).

DiffDock generates five poses per molecule and ranks them by pose confidence. The code saves the top pose as an SDF structure file and displays its confidence. Higher confidence ranks the pose more favorably within this docking workflow.

In [ ]:
docked, docking_envelopes = [], {}
for record in ([reference] + selected if selected else []):
    payload = {
        'protein': receptor_pdb,
        'ligand': record['smiles'],
        'ligand_file_type': 'txt',
        'num_poses': 5,
        'time_divisions': 20,
        'steps': 18,
        'save_trajectory': False,
    }
    envelope = (w.read_json(PREPARED / ('diffdock_' + record['candidate_id'] + '.json'))
                if USE_PREPARED else w.call_nim('diffdock', payload, RESULTS))
    response = envelope['response']
    pose = response['ligand_positions'][0]
    (RESULTS / (record['candidate_id'] + '_diffdock.sdf')).write_text(pose)
    docked.append(dict(record, pose_confidence=response['position_confidence'][0], pose_sdf=pose))
    docking_envelopes[record['candidate_id']] = envelope
display(pd.DataFrame(docked).drop(columns=['pose_sdf', 'smiles'], errors='ignore').round(3))
if not selected:
    print('No eligible analogues; docking skipped.')


### Select One Analogue and Inspect Its Docking Pose

Choose the analogue with the highest top-pose confidence for Boltz-2 evaluation. Gefitinib is retained as the reference rather than a candidate.

The viewer shows EGFR as a **grey cartoon**, experimental gefitinib as **green wireframe**, and the selected analogue as **orange sticks**. Inspect whether the candidate occupies the same pocket and how its orientation differs.

In [ ]:
import math
valid_docked = [row for row in docked if row['smiles'] != reference['smiles']
                and isinstance(row.get('pose_confidence'), (int, float))
                and math.isfinite(row['pose_confidence'])]
invalid_docked = [row['candidate_id'] for row in docked
                  if row['smiles'] != reference['smiles'] and row not in valid_docked]
if invalid_docked:
    print('Excluded from ranking: DiffDock returned no finite confidence:', invalid_docked)
w.save_json(RESULTS / 'docking_ranking_exclusions.json', invalid_docked)
chosen = max(valid_docked, key=lambda row: row['pose_confidence'], default=None)
if chosen is not None:
    w.save_json(RESULTS / 'chosen_candidate.json', chosen)
    print('Chosen for affinity:', chosen['candidate_id'], chosen['smiles'])
    viewer = w.show_complex(receptor_pdb, reference_sdf, chosen['pose_sdf'],
                            stem=FIGURES / 'diffdock_pose')
else:
    print('No docked analogue is available for affinity evaluation.')


## 4. Evaluate with Boltz-2

[NVIDIA Build](https://build.nvidia.com/mit/boltz2/deploy) · [Boltz-2 1.10 API](https://docs.nvidia.com/nim/bionemo/boltz2/1.10.0/inference)

Boltz-2 uses the EGFR sequence, its MSA, and a ligand SMILES string to predict a protein–ligand complex. Setting `predict_affinity=True` also requests affinity scores. Only the selected analogue is submitted for live candidate evaluation.

### Load the Gefitinib Reference

Load the saved gefitinib score for comparison. When preparing a new dataset, the code calculates this reference once if the file is missing and saves it. The reference uses the same EGFR sequence, alignment, and prediction settings as the candidate.

In [ ]:
reference_results = None
if chosen is not None:
    reference_file = DATA / 'reference_results.json'
    if not reference_file.exists():
        reference_envelope = w.call_nim('boltz2', reference_payload, ROOT / 'results/reference')
        score = w.affinity(reference_envelope['response'], reference['smiles'])
        score.update(input_hash=w.fingerprint(w.context(reference_payload)),
                     image=reference_envelope['image'], source='precomputed gefitinib')
        w.save_json(reference_file, {'gefitinib': score})
    reference_results = w.load_reference(reference_payload)
    reference_results['gefitinib']
else:
    print('No candidate selected; reference affinity evaluation skipped.')


### Predict the Selected Complex and Compare Affinity Scores

The request contains a protein entry under `polymers` and the selected molecule under `ligands`. The response provides complex coordinates and affinity scores. The code checks that the result belongs to the selected molecule, then saves the comparison table and predicted complex in mmCIF format.

Inspect **affinity_pic50** (higher means stronger predicted affinity), **affinity_probability_binary** (the predicted binding probability, from 0 to 1), and the model's structural-confidence score alongside the gefitinib reference.

In [ ]:
candidate_envelope, evaluated = None, None
if chosen is not None:
    candidate_payload = w.boltz_payload(chosen['smiles'], sequence, msa)
    candidate_envelope = (w.read_json(PREPARED / 'boltz2.json') if USE_PREPARED else
        w.call_nim('boltz2', candidate_payload, RESULTS, candidate=True))
    actual_smiles = candidate_envelope['request']['ligands'][0]['smiles']
    evaluated = w.affinity(candidate_envelope['response'], actual_smiles)
    assert evaluated['candidate_id'] == chosen['candidate_id'], 'Candidate identities must match.'
    comparison = pd.DataFrame([
        dict(reference_results['gefitinib'], name='Gefitinib'),
        dict(evaluated, name='Selected analogue'),
    ])
    columns = ['name', 'candidate_id', 'affinity_pic50', 'affinity_pred_value',
               'affinity_probability_binary', 'structural_confidence']
    comparison.to_csv(RESULTS / 'affinity_comparison.csv', index=False)
    display(comparison[columns])
    raw_complex = candidate_envelope['response']['structures'][0]['structure']
    (RESULTS / 'boltz2_complex.cif').write_text(raw_complex)
else:
    print('No candidate selected; Boltz-2 skipped.')


### View the Boltz-2 Complex in the Experimental Reference Frame

Align the predicted EGFR structure to experimental EGFR using matching **Cα atoms**, then apply the same transformation to the predicted ligand. The reported Cα RMSD describes the protein alignment in ångströms.

The overlay uses the same grey, green, and orange styling as the docking view. Compare the Boltz-2 ligand position with the experimental gefitinib pose and the earlier DiffDock result. Interactive HTML and PNG views are saved together.

In [ ]:
if candidate_envelope is not None:
    aligned_sdf, ca_rmsd = w.align_boltz_ligand(candidate_envelope['response'], actual_smiles, receptor_pdb)
    (RESULTS / 'boltz2_aligned_ligand.sdf').write_text(aligned_sdf)
    print('Protein alignment CA RMSD:', round(ca_rmsd, 3), 'Å')
    viewer = w.show_complex(receptor_pdb, reference_sdf, aligned_sdf,
                            stem=FIGURES / 'boltz2_complex')
else:
    print('No predicted candidate complex to display.')


## 5. Interpretation and Export

### Interpret the Current Run

Display the comparison already calculated in Section 4. The example probability filter is applied to the current results.


In [ ]:
if evaluated is not None:
    summary = comparison[['name', 'affinity_pic50', 'affinity_probability_binary',
                          'structural_confidence', 'cLogP', 'QED']].copy()
    summary['Example pass (probability >= 0.5)'] = summary['affinity_probability_binary'] >= 0.5
    display(summary.round(3).set_index('name').T)


Which molecules pass the example filter? Does higher predicted affinity come with higher cLogP or lower confidence? The complex views help assess contacts and clashes; protein-alignment RMSD describes the overlay rather than binding strength.

### Save Representative Results

Save a complete example pathway to `prepared/`: the generated library, selected candidates, docking responses, chosen molecule, and Boltz-2 response. These files allow a later run to resume from saved results with matching molecule identities.

The figures under `figures/` show molecular structures, the docking pose, and the predicted complex. Their PNG versions can be inserted into `BioNeMo_with_Agent.md`; HTML versions retain interactive views.

In [ ]:
if chosen is not None and candidate_envelope is not None:
    w.save_json(PREPARED / 'genmol.json', generation)
    w.save_json(PREPARED / 'candidates.json', selected)
    for candidate_id, envelope in docking_envelopes.items():
        w.save_json(PREPARED / ('diffdock_' + candidate_id + '.json'), envelope)
    w.save_json(PREPARED / 'boltz2.json', candidate_envelope)
    w.save_json(PREPARED / 'chosen_candidate.json', chosen)
    w.save_json(PREPARED / 'example_manifest.json', {
        'pdb': '2ITY', 'chain': 'A', 'reference': reference,
        'chosen_candidate': {key: chosen[key] for key in ('candidate_id', 'smiles')},
        'candidate_scores': evaluated,
        'figures': ['figures/genmol_molecules.png', 'figures/diffdock_pose.png',
                    'figures/boltz2_complex.png'],
    })
    print('Saved figures, prepared results, affinity CSV, SDF poses, and predicted complex.')
else:
    print('No complete pathway to save. Existing prepared results are preserved.')


### Conclusion and Next Steps

The current table and complex views bring together pose quality, confidence, affinity, and molecular-property trade-offs. If affinity improves while cLogP increases, solubility is a useful next question.

- Compare pose and score consistency across independent runs after the session.
- Assess solubility, selectivity, and synthetic accessibility.
- Compare EGFR inhibition and cellular activity with gefitinib.


## Release GPU Resources (Workshop VM Only)

Before switching to another GPU course, save this notebook, set `RELEASE_GPU = True` in the cell below, and run it to stop the BioNeMo models. Keep it `False` when continuing with the agent exercises.


In [ ]:
RELEASE_GPU = False  # Set True only when leaving BioNeMo.
if RELEASE_GPU:
    import shutil, subprocess
    if shutil.which("bootcamp-resources"):
        subprocess.run(["bootcamp-resources", "stop", "--confirm"], check=True)
    else:
        print("Self-study: stop your own NIMs with docker compose stop; see README.md.")


Check that `gpu_processes` is empty before starting the next course:


In [ ]:
import shutil, subprocess
if shutil.which("bootcamp-resources"):
    subprocess.run(["bootcamp-resources", "status"], check=True)
else:
    print("Self-study: check nvidia-smi and the NIM readiness URLs in README.md.")


Select **Kernel → Shut Down Kernel** to close this notebook's kernel.
